In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install gitsource

In [3]:
from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()


In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(len(documents))

72


In [5]:
!pip install minsearch


In [6]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [7]:
results = index.search("How does the agentic loop keep calling the model until it stops?")

print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [8]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.5 MB/s eta 0:00:00


In [9]:
import os
from groq import Groq

client = Groq(api_key="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")

def build_context(results):
    context = ""
    for doc in results:
        context += f"File: {doc['filename']}\n{doc['content']}\n\n"
    return context

def rag(query):
    # 1. search
    results = index.search(query, num_results=2)
    
    # 2. build prompt
    context = build_context(results)
    prompt = f"""Answer the question based on the context below.

Context:
{context}

Question: {query}
"""
    
    # 3. ask LLM
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response.choices[0].message.content
    input_tokens = response.usage.prompt_tokens
    
    return answer, input_tokens

In [10]:
query = "How does the agentic loop keep calling the model until it stops?"
answer, tokens = rag(query)

print("Input tokens:", tokens)
print("\nAnswer:", answer)

Input tokens: 3671

Answer: The agentic loop keeps calling the model until it stops by using a `while` loop that continues to execute until a condition is met. In this case, the condition is that the model returns a response without any function calls. As long as the response contains a function call, the loop continues to run, sending another message to the model with the updated conversation history. This process repeats until the model returns a response with no function calls, at which point the loop breaks and the model is considered done.

In more detail, here's what happens in each iteration of the loop:

1. The model is sent a new message with the updated conversation history (i.e., the messages that have been exchanged so far).
2. The model responds with a new message that may contain function calls, messages, or other content.
3. The loop checks if the response contains any function calls. If it does, the loop continues to run.
4. If the response contains no function calls, t

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [12]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [13]:
def rag_chunked(query):
    results = chunk_index.search(query, num_results=3)
    
    context = build_context(results)
    prompt = f"""Answer the question based on the context below.

Context:
{context}

Question: {query}
"""
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response.choices[0].message.content
    input_tokens = response.usage.prompt_tokens
    
    return answer, input_tokens

query = "How does the agentic loop keep calling the model until it stops?"
answer, tokens = rag_chunked(query)

print("Input tokens:", tokens)

Input tokens: 1488


In [14]:
import json

messages = [
    {"role": "system", "content": "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."},
    {"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}
]

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course materials for information about a topic.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"]
        }
    }
}

def search(query: str) -> str:
    results = chunk_index.search(query, num_results=1)
    return build_context(results)

In [15]:
search_count = 0
max_iterations = 5

while True:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        tools=[search_tool]
    )
    
    msg = response.choices[0].message
    messages.append(msg)
    
    if msg.tool_calls and search_count < max_iterations:
        for tool_call in msg.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = search(args["query"])
            search_count += 1
            print(f"Search #{search_count}: {args['query']}")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
    else:
        print("\nFinal answer:", msg.content)
        break

print(f"\nTotal searches: {search_count}")

Search #1: agentic loop vs RAG
Search #2: agentic loop definition
Search #3: RAG plain
Search #4: agentic loop vs plain RAG

Final answer: Based on the results of the searches, it appears that the agentic loop is a type of framework or architecture used for building conversational AI systems. It's designed to be modular, allowing developers to easily swap out different components and iterate on their chatbots. The agentic loop is similar to a plain RAG (Retrieval-Augmented Generation) model, but it has additional features that make it more flexible and powerful.

One key difference between the agentic loop and a plain RAG model is the use of vector search instead of keyword search. This allows the model to retrieve relevant information based on semantic similarity, rather than just matching keywords. Another difference is the ability to easily continue a conversation by passing the previous messages as input for the next call.

Overall, the agentic loop seems to be a powerful tool for 